In [1]:
import torch
import torchtext
import numpy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils import data

In [2]:
torch.__version__

'2.0.1+cu118'

In [3]:
torch.cuda.is_available()

True

In [4]:
torchtext.__version__

'0.15.2'

In [5]:
train_data = torchtext.datasets.IMDB(split='train')
train_iter = iter(train_data)

In [6]:
next(train_iter)

(1,
 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betwee

In [7]:
# next(train_iter)

In [8]:
train_datas = [(label,text) for (label,text) in train_iter]

In [9]:
train_datas[:5]

[(1,
  '"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn\'t matter what one\'s political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn\'t true. I\'ve seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don\'t exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris in sight. And those pretentious indie movies like The Brown Bunny, in which we\'re treated to the site of Vincent Gallo\'s throbbing johnson, but not a trace of pink visible on Chloe Sevigny. Before crying (or implying) "double-standard" in matters of nudity, the mentally obtuse should take into account one unavoidably obvious anatomical difference between men and women: there are no genitals on display when actresses appears nude, an

In [10]:
len(train_datas)

24999

In [11]:
torchtext.datasets.IMDB()

(ShardingFilterIterDataPipe, ShardingFilterIterDataPipe)

In [12]:
train_iter,test_iter = torchtext.datasets.IMDB()

In [13]:
# 导入分词工具
from torchtext.data.utils import get_tokenizer
# 导入此表的工具，从迭代器中构建词汇表工具
from torchtext.vocab import build_vocab_from_iterator

In [14]:
# 分词工具初始化
tokenizer = get_tokenizer('basic_english')
tokenizer

<function torchtext.data.utils._basic_english_normalize(line)>

In [15]:
tokenizer('I was just an early teen when this smoked fish hit the U.S.')

['i',
 'was',
 'just',
 'an',
 'early',
 'teen',
 'when',
 'this',
 'smoked',
 'fish',
 'hit',
 'the',
 'u',
 '.',
 's',
 '.']

In [16]:

def yield_tokens(data):
    for _,text in data:
        yield tokenizer(text)

# 转换分词的可迭代对象
train_tokens = yield_tokens(train_iter)
train_tokens

<generator object yield_tokens at 0x000002A4DBD3CCF0>

In [17]:
# next(train_tokens)

In [18]:
# 通过迭代器生成的词汇表，词汇表是一个映射，将单词转换为数字的索引
vocab = build_vocab_from_iterator(train_tokens,specials=['<pad>','<unk>'])
vocab.set_default_index(vocab["<unk>"])
vocab

Vocab()

In [19]:
vocab(['i','love','you'])

[13, 125, 26]

In [20]:
# 文本处理管道，文本-》分词--》数字的索引
text_pipeline = lambda x: vocab(tokenizer(x))

In [21]:
# lambda 匿名函数等价写法
# 和lambda完全等效
# def text_pipeline(x):
#     return vocab(tokenizer(x))

# def label_pipeline(label_text):
#     return int(label_text == "pos")

In [22]:
# 将label标签从文本转成0/1
# label_pipline = lambda label_text : int(label_text=='pos')
label_pipline = lambda label_text : int(label_text==1)

In [23]:
text_pipeline('i love you!')

[13, 125, 26, 36]

In [24]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

device

'cuda'

In [25]:
# 设置数据集加载器dataloader
from torch.utils.data import DataLoader

# 生产一个批次训练的数据
def collate_batch(batch):
    # label存储转换后的标签，text_list存储处理后的文本数据，offsets记录每个文本在text_list的位置
    label_list,text_list = [],[]
    for (_label,_text) in batch:
        label_list.append(label_pipline(_label))
        precess_text = torch.tensor(text_pipeline(_text),dtype=torch.int64)
        text_list.append(precess_text)
        # offsets.append(precess_text.size(0))

    label_list = torch.tensor(label_list)
    # 将评论长度统一，不足的填充0
    text_list = torch.nn.utils.rnn.pad_sequence(text_list)
    # text_list = torch.cat(text_list)
    # offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    
    return label_list.to(device),text_list.to(device)

In [26]:
from torchtext.data.functional import to_map_style_dataset
train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)

In [27]:
BATCH_SIZE = 64
# 数据加载器
train_dl = torch.utils.data.DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collate_batch
)
test_dl = torch.utils.data.DataLoader(
    test_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collate_batch
)

In [28]:
# for label,text in train_dl:
#     print(label)
#     break

In [29]:
vocal_size = len(vocab)
vocal_size

100684

In [30]:
# 创建LSTM循环神经网络单元
hidden_size = 128
class RNN_Encoder(nn.Module):
    # 输入的维度，隐藏层的大小
    def __init__(self,input_dim,hidden_size):
        super(RNN_Encoder,self).__init__()
        self.rnn_cell = nn.LSTMCell(input_dim,hidden_size)
    def forward(self,inputs):
        # input输入数据的形状sequence_length,batch_size,input_dim
        bz = inputs.shape[1]
        # ht为隐藏装动态
        ht = torch.zeros((bz,hidden_size)).to(device)
        # 增加单元隐藏状态
        ct = torch.zeros((bz,hidden_size)).to(device)
        for word in inputs:
            ht,ct = self.rnn_cell(word,(ht,ct))
        return ht

In [31]:
class RNN_Net(nn.Module):
    def __init__(self,vocal_size,embeding_dim,hidden_size):
        super(RNN_Net,self).__init__()
        # 创建一个嵌入层，固定输入的维度
        # 一段文字-》词汇表对应的索引-》转换为嵌入向量
        self.em = nn.Embedding(vocal_size,embeding_dim)
        # 实例化循环神经网络
        self.rnn = RNN_Encoder(embeding_dim,hidden_size)
        # 实例化全连接层
        self.fc1 = nn.Linear(hidden_size,64)
        self.fc2 = nn.Linear(64,2)

    def forward(self,inputs):
        x = self.em(inputs)
        x = self.rnn(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x

In [32]:
# 创建文本分类模型
class TextClassificationModel(nn.Module):
    def __init__(self,vocab_size,embed_dim,num_class):
        # 定义词汇表大小，嵌入维度embed_dim,分类的类别
        super(TextClassificationModel,self).__init__()

        # 实例化一层嵌入层。EmbeddingBag
        self.embedding = nn.EmbeddingBag(vocab_size,embed_dim,sparse=True)
        # 实例化全连接网络层来实现分类
        self.fc = nn.Linear(embed_dim,num_class)
    def forward(self,text,offsets):
        embedded = self.embedding(text,offsets)
        output = self.fc(embedded)
        return output

In [33]:
emsize = 100
num_class=2

In [34]:
# model = TextClassificationModel(vocal_size,emsize,num_class)
model = RNN_Net(vocal_size,emsize,128)
model = model.to(device)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\rehuanoo\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\rehuanoo\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "S:\PythonAILearningCourse\09-pytorch-deep-learning\venv\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "S:\PythonAILearningCourse\09-pytorch-deep-learning\venv\lib\site-packag

In [35]:
model

RNN_Net(
  (em): Embedding(100684, 100)
  (rnn): RNN_Encoder(
    (rnn_cell): LSTMCell(100, 128)
  )
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)

In [36]:
from torch.optim import lr_scheduler
loss_fn = nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.01)

exp_lr_scheduler = lr_scheduler.StepLR(optimizer,step_size=5,gamma=0.1)

In [37]:
next(enumerate(train_dl))

(0,
 (tensor([1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1,
          1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1,
          0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0], device='cuda:0'),
  tensor([[51013,   822,    14,  ...,    13,     6,    35],
          [   36,    15,    10,  ...,   227,   221,     7],
          [  443,    26,     6,  ...,   141,     7,     2],
          ...,
          [    0,     0,     0,  ...,     0,     0,     0],
          [    0,     0,     0,  ...,     0,     0,     0],
          [    0,     0,     0,  ...,     0,     0,     0]], device='cuda:0')))

In [38]:
def train(dataloader):
    total_acc,total_count,total_loss=0,0,0
    # 将模型设置训练模式
    model.train()
    for label,text in dataloader:
        predited_label = model(text)
        loss = loss_fn(predited_label,label)

        optimizer.zero_grad()
        loss.backward()
        # for name, param in model.named_parameters():
        #     if param.grad is not None:
        #         print(name, param.grad.norm())
        optimizer.step()
        with torch.no_grad():
            total_acc += (predited_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
            total_loss += loss.item()*label.size(0)
    return total_loss/total_count,total_acc/total_count
            

In [39]:
def test(dataloader):
    total_acc,total_count,total_loss=0,0,0
    # 将模型设置评估模式
    model.eval()
    with torch.no_grad():
        for idx,(label,text) in enumerate(dataloader):
            predited_label = model(text)
            loss = loss_fn(predited_label,label)
            
            total_acc += (predited_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
            total_loss += loss.item()*label.size(0)
    return total_loss/total_count,total_acc/total_count

In [40]:
# 训练多轮
epochs = 10
train_acc=[]
test_acc=[]
train_loss=[]
test_loss=[]

# 保存最佳的准确率
best_acc = 0.0
best_acc_index=0
for epoch in range(epochs):
    epoch_loss,epoch_acc = train(train_dl)
    epoch_test_loss,epoch_test_acc = test(test_dl)
    train_acc.append(epoch_acc)
    test_acc.append(epoch_test_acc)
    train_loss.append(epoch_loss)
    test_loss.append(epoch_test_loss)
    print('epoch:{},train_loss:{},train_acc{}'.format(epoch,epoch_loss,epoch_acc))
    print('epoch:{},test_loss:{},test_acc{}'.format(epoch,epoch_test_loss,epoch_test_acc))
    print('---------------------------------')
    # if epoch_test_acc > best_acc:
    #     print('保存最佳准确率模型',epoch_test_acc)
    #     best_acc = epoch_test_acc
    #     best_acc_index = best_acc_index+1
    #     # 保存模型结构，包含参数
    #     torch.save(model,'best_model.pth')
    #     # 保存参数
    #     # torch.save(model1.state_dict(),'best_model_weights'+str(best_acc_index)+'.pth')

em.weight tensor(0.0239, device='cuda:0')
rnn.rnn_cell.weight_ih tensor(0.4215, device='cuda:0')
rnn.rnn_cell.weight_hh tensor(0.1016, device='cuda:0')
rnn.rnn_cell.bias_ih tensor(0.0400, device='cuda:0')
rnn.rnn_cell.bias_hh tensor(0.0400, device='cuda:0')
fc1.weight tensor(0.3401, device='cuda:0')
fc1.bias tensor(0.1333, device='cuda:0')
fc2.weight tensor(0.2346, device='cuda:0')
fc2.bias tensor(0.2619, device='cuda:0')
em.weight tensor(0.0147, device='cuda:0')
rnn.rnn_cell.weight_ih tensor(0.2621, device='cuda:0')
rnn.rnn_cell.weight_hh tensor(0.0631, device='cuda:0')
rnn.rnn_cell.bias_ih tensor(0.0256, device='cuda:0')
rnn.rnn_cell.bias_hh tensor(0.0256, device='cuda:0')
fc1.weight tensor(0.2142, device='cuda:0')
fc1.bias tensor(0.0881, device='cuda:0')
fc2.weight tensor(0.1552, device='cuda:0')
fc2.bias tensor(0.1873, device='cuda:0')
em.weight tensor(0.0087, device='cuda:0')
rnn.rnn_cell.weight_ih tensor(0.1559, device='cuda:0')
rnn.rnn_cell.weight_hh tensor(0.0373, device='cuda:

KeyboardInterrupt: 